# K-channel sub-network brain figures (Shi-style channel-to-channel on cortex)

**Builds:**
- `fig_k23_subnetwork.png`, `fig_k16_subnetwork.png`, `fig_k12_subnetwork.png`, `fig_k8_subnetwork.png` — standalone panels.
- `fig_k_subnetwork_4panel.png` — 1×4 composite, K=23 (left) → K=8 (right).

**Source data:**
- Signed differential pair matrix: `research/xai/channel_reduction/st_hbo/channel_pair_matrix_diff.npy` (convention: `mean(GAD) − mean(HC)`; **positive → GAD>HC**, **negative → HC>GAD**).
- K-subsets: `research/xai/channel_reduction/run.json → channel_ablation_test_points.{primary, alternate_1, alternate_2}` (the *same* subsets used in P1.7 channel-ablation experiment).

**Edge rule:** top-N=8 absolute-value edges whose **both endpoints** lie in the K-subset.

**Colors:** red = HC > GAD (negative diff); blue = GAD > HC (positive diff). Per user feedback 2026-05-14.

**Anchor cell:** ST × HbO × LOSO × mt=2 (paper headline).


In [19]:
# Cell 1 — Imports, paths, MNE 3D backend
# PATH UPDATE 2026-05-15: figures/ was reorganised into methods/ + results/ categories.
# This notebook writes to results/k_subnetwork/. Single-N preview (cells 5+6) goes
# into results/k_subnetwork/_scratch/ to avoid clobbering batched N{25,15,8}/ outputs.
import os, json
os.environ["PYVISTA_OFF_SCREEN"] = "true"
os.environ["MPLBACKEND"] = "Agg"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import rcParams
import imageio.v3 as iio
import pyvista as pv
import mne

mne.set_log_level("ERROR")
rcParams["font.family"] = ["DejaVu Sans", "Helvetica", "Arial", "sans-serif"]
rcParams["savefig.dpi"] = 300

REPO   = Path('/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method')
ELC    = REPO / 'data' / 'brainproducts-RNP-BA-128-custom.elc'
GNG_DIR = REPO / 'data' / 'raw' / 'anxiety' / 'EA062' / 'GNG'
XAI_DIR = REPO / 'research' / 'xai' / 'channel_reduction' / 'st_hbo'
RUN_JSON = REPO / 'research' / 'xai' / 'channel_reduction' / 'run.json'
# Canonical output root (post-2026-05-15 reorg)
FIG_DIR     = REPO / 'research' / 'paper-materials' / 'figures'
BATCH_ROOT  = FIG_DIR / 'results' / 'k_subnetwork'         # cell 7 writes N25/N15/N8 here
SCRATCH     = BATCH_ROOT / '_scratch'                       # cells 5+6 single-N preview lands here
INTERM      = SCRATCH                                       # alias for cell-5 intermediate renders
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

mne.viz.set_3d_backend("notebook")
print("MNE:", mne.__version__, "| PyVista:", pv.__version__, "| backend:", mne.viz.get_3d_backend())
print("XAI dir exists:", XAI_DIR.exists(), "| run.json exists:", RUN_JSON.exists())
print("BATCH_ROOT:", BATCH_ROOT.relative_to(REPO))
print("SCRATCH   :", SCRATCH.relative_to(REPO))

MNE: 1.12.0 | PyVista: 0.47.3 | backend: notebook
XAI dir exists: True | run.json exists: True
BATCH_ROOT: research/paper-materials/figures/results/k_subnetwork
SCRATCH   : research/paper-materials/figures/results/k_subnetwork/_scratch


In [7]:
# Cell 2 — fsaverage + channel positions
#   Pipeline copied from notebook_xai_brain_figures.ipynb (cells 2 + 6):
#   read_raw_nirx -> set_montage -> optical_density -> beer_lambert -> pick(hbo)
#   -> head_to_MRI -> cluster contraction 0.85 -> Z shift -20 mm
#
# CHANGE (2026-05-14): lift dots+edges OUTWARD radially so they sit ABOVE the
# cortex surface and cannot be occluded by the pial mesh.  Used together with
# translucent cortex (alpha=0.35) in cell 4.
fs_root       = mne.datasets.fetch_fsaverage(verbose=False)
subjects_dir  = Path(fs_root).parent

raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)
montage = mne.channels.read_custom_montage(str(ELC), head_size=0.0825)
raw.set_montage(montage)
raw_od    = mne.preprocessing.nirs.optical_density(raw, verbose=False)
raw_haemo = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=6.0)
info      = raw_haemo.copy().pick_types(fnirs='hbo', verbose=False).info

trans          = mne.read_trans(str(subjects_dir / "fsaverage" / "bem" / "fsaverage-trans.fif"))
head_positions = np.array([ch["loc"][:3] for ch in info["chs"]])
mri_positions_mm = mne.transforms.apply_trans(trans["trans"], head_positions) * 1000.0

cluster_centre = mri_positions_mm.mean(axis=0)
shrink_factor  = 0.85
base_shrunk    = cluster_centre + (mri_positions_mm - cluster_centre) * shrink_factor
mri_pos_dots_raw   = base_shrunk.copy()
mri_pos_dots_raw[:, 2] -= 20.0  # original "inside the skull" position

# Lift OUTWARD along radial direction so dots+lines sit clearly above the pial surface.
DOT_LIFT_MM  = 12.0   # 10-15 mm range works well; visibility vs anatomical readability trade-off
EDGE_LIFT_MM = 12.0

radial = mri_pos_dots_raw - cluster_centre
radial_unit = radial / (np.linalg.norm(radial, axis=1, keepdims=True) + 1e-9)
mri_pos_dots = mri_pos_dots_raw + DOT_LIFT_MM  * radial_unit
edge_pos     = mri_pos_dots_raw + EDGE_LIFT_MM * radial_unit  # tubes use same anchor

CHANNEL_NAMES = ['S1_D1','S1_D3','S2_D2','S2_D1','S2_D5','S3_D1','S3_D3','S3_D4','S3_D6',
                 'S4_D4','S4_D5','S4_D7','S5_D2','S5_D5','S5_D8','S6_D3','S6_D6',
                 'S7_D4','S7_D6','S7_D7','S8_D5','S8_D7','S8_D8']
N_CH  = len(CHANNEL_NAMES)
CH_IDX = {n: i for i, n in enumerate(CHANNEL_NAMES)}
assert info["ch_names"][0].startswith("S1_D1"), info["ch_names"][:3]
print(f"n_channels={N_CH}  | DOT_LIFT_MM={DOT_LIFT_MM}  | EDGE_LIFT_MM={EDGE_LIFT_MM}")
print(f"S1_D1 raw   (MRI mm): {mri_pos_dots_raw[CH_IDX['S1_D1']].round(1)}")
print(f"S1_D1 lifted(MRI mm): {mri_pos_dots[CH_IDX['S1_D1']].round(1)}  -> radial dist from centroid: {np.linalg.norm(mri_pos_dots[CH_IDX['S1_D1']] - cluster_centre):.1f} mm")

n_channels=23  | DOT_LIFT_MM=12.0  | EDGE_LIFT_MM=12.0
S1_D1 raw   (MRI mm): [-15.6  66.6  -8.1]
S1_D1 lifted(MRI mm): [-19.2  70.1 -18.9]  -> radial dist from centroid: 66.9 mm


/tmp/ipykernel_4145305/1010860334.py:12: RuntimeWarning: MNE has not been tested with Aurora version 2021.9.0-20-g15526401-dirty
  raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)


In [3]:
# Cell 3 — Load signed diff matrix + K-subsets + sanity-check sign convention
diff = np.load(XAI_DIR / 'channel_pair_matrix_diff.npy')
assert diff.shape == (N_CH, N_CH), diff.shape
assert np.allclose(diff, diff.T), "diff must be symmetric"

with open(RUN_JSON) as f:
    run = json.load(f)
tp = run['channel_ablation_test_points']
K_SUBSETS = {
    23: CHANNEL_NAMES,                  # full set
    16: tp['alternate_2']['channels'],  # K=16
    12: tp['primary']['channels'],      # K=12 (paper headline)
     8: tp['alternate_1']['channels'],  # K=8
}
for K, chans in K_SUBSETS.items():
    assert len(chans) == K, (K, len(chans))
K_IDX = {K: np.array([CH_IDX[c] for c in chans]) for K, chans in K_SUBSETS.items()}

# Sanity (matches project_channel_reduction_analysis.md):
#   S2_D1-S3_D3 GAD>HC -> positive diff
#   S2_D1-S3_D6 HC>GAD -> negative diff
for a, b, expected in [('S2_D1','S3_D3','+ (GAD>HC)'),
                       ('S2_D1','S3_D6','- (HC>GAD)'),
                       ('S2_D1','S6_D6','+ (GAD>HC)')]:
    v = diff[CH_IDX[a], CH_IDX[b]]
    print(f"  diff[{a},{b}] = {v:+.5f}   expected {expected}")
print(f"K_SUBSETS: " + ", ".join(f"K={K}->{len(v)} chans" for K, v in K_SUBSETS.items()))

  diff[S2_D1,S3_D3] = +0.00549   expected + (GAD>HC)
  diff[S2_D1,S3_D6] = -0.00608   expected - (HC>GAD)
  diff[S2_D1,S6_D6] = +0.00577   expected + (GAD>HC)
K_SUBSETS: K=23->23 chans, K=16->16 chans, K=12->12 chans, K=8->8 chans


In [15]:
# Cell 4 — Helpers: top-N edge selection + render-one-panel
# CHANGE (2026-05-14): Brain alpha 1.0 -> 0.35 (translucent cortex).
# CHANGE (2026-05-15): SWAP color convention: RED=GAD>HC, BLUE=HC>GAD (Shi-consistent).
# CHANGE (2026-05-15): Edge-selection rule = GLOBAL top-N ∩ K (top-decile=N=25, matches
# research/xai/channel_reduction/figures/subnetwork_top_decile.png). Tube radius scales
# with |M_diff| so strongest edges stand out in the dense K=23 panel.
TOP_N = 25
CORTEX_ALPHA = 0.35
CORTEX_STYLE = 'low_contrast'

ROI_COLOR = {
    'vmpfc':    (0.0, 0.0, 1.0),
    'dmpfc':    (0.0, 1.0, 0.0),
    'dlpfc':    (1.0, 0.0, 0.0),
    'default':  (1.0, 0.647, 0.0),
}
ROI_OF_IDX = {}
for i in [0, 3, 2]:                                 ROI_OF_IDX[i] = 'vmpfc'
for i in [7, 17, 18, 9, 19, 10, 11, 21]:            ROI_OF_IDX[i] = 'dmpfc'
for i in [6, 8, 16, 15, 13, 14, 22, 20]:            ROI_OF_IDX[i] = 'dlpfc'
for i in range(N_CH):
    ROI_OF_IDX.setdefault(i, 'default')

EDGE_COLOR_GAD_GT_HC = '#E74C3C'  # RED  : positive diff (GAD > HC, anxiety-associated)
EDGE_COLOR_HC_GT_GAD = '#5577CC'  # BLUE : negative diff (HC > GAD, healthy-dominant)

DOT_RADIUS_MM      = 3.2
DOT_NONK_RADIUS_MM = 2.0
# Tube radius scales between MIN..MAX by |M_diff| rank within the global top-N set.
TUBE_RADIUS_MIN_MM = 0.7
TUBE_RADIUS_MAX_MM = 1.7
EDGE_OPACITY       = 0.95
NONK_DOT_ALPHA     = 0.35
LABEL_UP_MM        = 6.0

# Pre-compute the global top-N ranking ONCE (same edges considered for every K).
def select_global_top_n(diff_matrix, n):
    pairs = []
    for i in range(diff_matrix.shape[0]):
        for j in range(i + 1, diff_matrix.shape[1]):
            pairs.append((i, j, float(diff_matrix[i, j])))
    pairs.sort(key=lambda e: abs(e[2]), reverse=True)
    return pairs[:n]

GLOBAL_TOP = select_global_top_n(diff, TOP_N)
abs_vals = np.array([abs(v) for (_, _, v) in GLOBAL_TOP])
ABS_MIN, ABS_MAX = abs_vals.min(), abs_vals.max()

def radius_for(v):
    """Linear mapping |v| -> [TUBE_RADIUS_MIN_MM, TUBE_RADIUS_MAX_MM]."""
    norm = (abs(v) - ABS_MIN) / (ABS_MAX - ABS_MIN + 1e-12)
    return TUBE_RADIUS_MIN_MM + norm * (TUBE_RADIUS_MAX_MM - TUBE_RADIUS_MIN_MM)

def filter_by_k(edges, k_indices):
    """Keep only edges where both endpoints are in k_indices."""
    k_set = set(int(i) for i in k_indices)
    return [(i, j, v) for (i, j, v) in edges if i in k_set and j in k_set]

def render_k_panel(K, save_stem):
    k_idx = K_IDX[K]
    k_set = set(int(i) for i in k_idx)
    edges = filter_by_k(GLOBAL_TOP, k_idx)

    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex=CORTEX_STYLE, alpha=CORTEX_ALPHA, units='mm', size=(1100, 950),
    )
    for i in range(N_CH):
        in_K  = (i in k_set)
        color = ROI_COLOR[ROI_OF_IDX[i]] if in_K else (0.6, 0.6, 0.6)
        alpha = 1.0 if in_K else NONK_DOT_ALPHA
        r     = DOT_RADIUS_MM if in_K else DOT_NONK_RADIUS_MM
        sphere = pv.Sphere(center=mri_pos_dots[i], radius=r)
        b.plotter.add_mesh(sphere, color=color, opacity=alpha, name=f"dot_{i}")
    label_pos = mri_pos_dots[list(k_idx)] + np.array([0, 0, LABEL_UP_MM])
    label_txt = [str(int(i) + 1) for i in k_idx]
    b.plotter.add_point_labels(
        label_pos, label_txt,
        font_size=22, text_color="black",
        point_color="white", point_size=1,
        show_points=False, always_visible=True, shape=None, margin=0,
        name=f"labels_{K}",
    )
    for (i, j, v) in edges:
        line = pv.Line(edge_pos[i], edge_pos[j], resolution=24).tube(radius=radius_for(v), n_sides=20)
        color = EDGE_COLOR_GAD_GT_HC if v > 0 else EDGE_COLOR_HC_GT_GAD
        b.plotter.add_mesh(line, color=color, opacity=EDGE_OPACITY, name=f"edge_{i}_{j}")
    try: b.show_view(view='rostral')
    except Exception: b.show_view()
    p = INTERM / f"{save_stem}.png"
    b.save_image(str(p), mode='rgba')
    b.close()
    return p, edges

print(f"Edge rule = GLOBAL top-{TOP_N} ∩ K (top-decile of {N_CH*(N_CH-1)//2} pairs)")
print(f"Tube radius scaling: |M_diff| ∈ [{ABS_MIN:.5f}, {ABS_MAX:.5f}] -> r ∈ [{TUBE_RADIUS_MIN_MM}, {TUBE_RADIUS_MAX_MM}] mm")
print(f"Colors (Shi-consistent): RED={EDGE_COLOR_GAD_GT_HC} (GAD>HC), BLUE={EDGE_COLOR_HC_GT_GAD} (HC>GAD)")

Edge rule = GLOBAL top-25 ∩ K (top-decile of 253 pairs)
Tube radius scaling: |M_diff| ∈ [0.00371, 0.00669] -> r ∈ [0.7, 1.7] mm
Colors (Shi-consistent): RED=#E74C3C (GAD>HC), BLUE=#5577CC (HC>GAD)


In [16]:
# Cell 5 — Render the 4 K panels (K=23 / 16 / 12 / 8) and print surviving edges per panel
import sys, time
panel_paths = {}
panel_edges = {}
t0 = time.time()
for K in [23, 16, 12, 8]:
    print(f"\n=== Rendering K={K} ===")
    sys.stdout.flush()
    p, edges = render_k_panel(K, f"k_subnet_K{K:02d}")
    panel_paths[K] = p
    panel_edges[K] = edges
    print(f"  Saved: {p.name}")
    print(f"  Surviving edges (global top-{TOP_N} ∩ K={K}): {len(edges)} / {TOP_N}")
    for (i, j, v) in edges:
        sign = "GAD>HC" if v > 0 else "HC>GAD"
        col  = "RED" if v > 0 else "BLUE"
        print(f"    {CHANNEL_NAMES[i]:>6s} ↔ {CHANNEL_NAMES[j]:>6s}  diff={v:+.5f}  {sign:8s} ({col})  tube_r={radius_for(v):.2f}mm")
print(f"\nTotal render time: {time.time()-t0:.1f}s")


=== Rendering K=23 ===


  Saved: k_subnet_K23.png
  Surviving edges (global top-25 ∩ K=23): 25 / 25
     S2_D2 ↔  S8_D5  diff=-0.00669  HC>GAD   (BLUE)  tube_r=1.70mm
     S2_D1 ↔  S3_D6  diff=-0.00608  HC>GAD   (BLUE)  tube_r=1.50mm
     S2_D1 ↔  S6_D6  diff=+0.00577  GAD>HC   (RED)  tube_r=1.39mm
     S2_D1 ↔  S3_D3  diff=+0.00549  GAD>HC   (RED)  tube_r=1.30mm
     S3_D1 ↔  S4_D4  diff=+0.00548  GAD>HC   (RED)  tube_r=1.29mm
     S2_D1 ↔  S4_D4  diff=+0.00532  GAD>HC   (RED)  tube_r=1.24mm
     S2_D2 ↔  S8_D7  diff=-0.00513  HC>GAD   (BLUE)  tube_r=1.18mm
     S3_D4 ↔  S4_D7  diff=+0.00488  GAD>HC   (RED)  tube_r=1.09mm
     S4_D4 ↔  S7_D7  diff=+0.00484  GAD>HC   (RED)  tube_r=1.08mm
     S4_D7 ↔  S6_D3  diff=-0.00476  HC>GAD   (BLUE)  tube_r=1.05mm
     S1_D1 ↔  S6_D3  diff=-0.00473  HC>GAD   (BLUE)  tube_r=1.04mm
     S2_D1 ↔  S8_D5  diff=+0.00459  GAD>HC   (RED)  tube_r=1.00mm
     S3_D1 ↔  S6_D3  diff=+0.00459  GAD>HC   (RED)  tube_r=0.99mm
     S2_D2 ↔  S2_D1  diff=-0.00458  HC>GAD   (BLUE)  tube_r=0

  Saved: k_subnet_K16.png
  Surviving edges (global top-25 ∩ K=16): 16 / 25
     S2_D2 ↔  S8_D5  diff=-0.00669  HC>GAD   (BLUE)  tube_r=1.70mm
     S2_D1 ↔  S3_D6  diff=-0.00608  HC>GAD   (BLUE)  tube_r=1.50mm
     S2_D1 ↔  S6_D6  diff=+0.00577  GAD>HC   (RED)  tube_r=1.39mm
     S3_D1 ↔  S4_D4  diff=+0.00548  GAD>HC   (RED)  tube_r=1.29mm
     S2_D1 ↔  S4_D4  diff=+0.00532  GAD>HC   (RED)  tube_r=1.24mm
     S2_D2 ↔  S8_D7  diff=-0.00513  HC>GAD   (BLUE)  tube_r=1.18mm
     S3_D4 ↔  S4_D7  diff=+0.00488  GAD>HC   (RED)  tube_r=1.09mm
     S4_D7 ↔  S6_D3  diff=-0.00476  HC>GAD   (BLUE)  tube_r=1.05mm
     S1_D1 ↔  S6_D3  diff=-0.00473  HC>GAD   (BLUE)  tube_r=1.04mm
     S2_D1 ↔  S8_D5  diff=+0.00459  GAD>HC   (RED)  tube_r=1.00mm
     S3_D1 ↔  S6_D3  diff=+0.00459  GAD>HC   (RED)  tube_r=0.99mm
     S2_D2 ↔  S2_D1  diff=-0.00458  HC>GAD   (BLUE)  tube_r=0.99mm
     S5_D2 ↔  S7_D6  diff=-0.00450  HC>GAD   (BLUE)  tube_r=0.97mm
     S4_D4 ↔  S7_D6  diff=+0.00440  GAD>HC   (RED)  tube_r=

  Saved: k_subnet_K12.png
  Surviving edges (global top-25 ∩ K=12): 11 / 25
     S2_D2 ↔  S8_D5  diff=-0.00669  HC>GAD   (BLUE)  tube_r=1.70mm
     S2_D1 ↔  S3_D6  diff=-0.00608  HC>GAD   (BLUE)  tube_r=1.50mm
     S2_D1 ↔  S6_D6  diff=+0.00577  GAD>HC   (RED)  tube_r=1.39mm
     S3_D1 ↔  S4_D4  diff=+0.00548  GAD>HC   (RED)  tube_r=1.29mm
     S2_D1 ↔  S4_D4  diff=+0.00532  GAD>HC   (RED)  tube_r=1.24mm
     S2_D2 ↔  S8_D7  diff=-0.00513  HC>GAD   (BLUE)  tube_r=1.18mm
     S1_D1 ↔  S6_D3  diff=-0.00473  HC>GAD   (BLUE)  tube_r=1.04mm
     S2_D1 ↔  S8_D5  diff=+0.00459  GAD>HC   (RED)  tube_r=1.00mm
     S3_D1 ↔  S6_D3  diff=+0.00459  GAD>HC   (RED)  tube_r=0.99mm
     S2_D2 ↔  S2_D1  diff=-0.00458  HC>GAD   (BLUE)  tube_r=0.99mm
     S4_D4 ↔  S7_D6  diff=+0.00440  GAD>HC   (RED)  tube_r=0.93mm

=== Rendering K=8 ===


  Saved: k_subnet_K08.png
  Surviving edges (global top-25 ∩ K=8): 5 / 25
     S2_D1 ↔  S3_D6  diff=-0.00608  HC>GAD   (BLUE)  tube_r=1.50mm
     S3_D1 ↔  S4_D4  diff=+0.00548  GAD>HC   (RED)  tube_r=1.29mm
     S2_D1 ↔  S4_D4  diff=+0.00532  GAD>HC   (RED)  tube_r=1.24mm
     S2_D1 ↔  S8_D5  diff=+0.00459  GAD>HC   (RED)  tube_r=1.00mm
     S3_D1 ↔  S6_D3  diff=+0.00459  GAD>HC   (RED)  tube_r=0.99mm

Total render time: 13.3s


In [17]:
# Cell 6 — Crop whitespace, save 4 standalones, build 1×4 composite (single-N PREVIEW)
# NOTE: This is the single-N developer preview. Canonical paper outputs come from cell 7
# (batch generator over N=25/15/8 into results/k_subnetwork/N{25,15,8}/).
# Cell 6 saves into SCRATCH so re-runs don't pollute the batched output folders.
def crop_white(img_path, margin=8):
    img = iio.imread(str(img_path))
    rgb = img[:, :, :3].astype(np.int32)
    sat = rgb.max(axis=2) - rgb.min(axis=2)
    is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
    ys, xs = np.where(is_content)
    if len(ys) == 0: return img
    y0, y1 = max(0, ys.min()-margin), min(img.shape[0], ys.max()+margin)
    x0, x1 = max(0, xs.min()-margin), min(img.shape[1], xs.max()+margin)
    return img[y0:y1, x0:x1]

cropped = {K: crop_white(panel_paths[K]) for K in [23, 16, 12, 8]}

def panel_title(K):
    n_surv = len(panel_edges[K])
    return f"K = {K}   ({n_surv}/{TOP_N} top-decile edges)"

for K, im in cropped.items():
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.imshow(im)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(panel_title(K), fontsize=14, fontweight='bold')
    p_out = SCRATCH / f"fig_k{K:02d}_subnetwork.png"
    fig.savefig(str(p_out), dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(str(p_out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  Saved standalone (scratch): {p_out.relative_to(REPO)}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, K in zip(axes, [23, 16, 12, 8]):
    ax.imshow(cropped[K])
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(panel_title(K), fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], color=EDGE_COLOR_GAD_GT_HC, lw=5, label='GAD > HC (anxiety-associated)'),
    Line2D([0],[0], color=EDGE_COLOR_HC_GT_GAD, lw=5, label='HC > GAD (healthy-dominant)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=12, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Discriminative sub-network on cortex — global top-{TOP_N} |M_diff| edges (line width ∝ |M_diff|)",
             fontsize=14, y=1.0)
plt.subplots_adjust(left=0.005, right=0.995, top=0.88, bottom=0.07, wspace=0.02)
out = SCRATCH / 'fig_k_subnetwork_4panel.png'
fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"\nSaved composite (scratch): {out.relative_to(REPO)}")
print(f"\nFinal artefacts in {SCRATCH.relative_to(REPO)}/:")
for p in sorted(SCRATCH.glob('fig_k*subnetwork*')):
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.



Saved composite: fig_k_subnetwork_4panel.png

Final artefacts in /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/paper-materials/figures:
  fig_k08_subnetwork.png  (773.1 KB)
  fig_k08_subnetwork.svg  (1232.2 KB)
  fig_k12_subnetwork.png  (853.7 KB)
  fig_k12_subnetwork.svg  (1332.5 KB)
  fig_k16_subnetwork.png  (917.5 KB)
  fig_k16_subnetwork.svg  (1399.3 KB)
  fig_k23_subnetwork.png  (920.3 KB)
  fig_k23_subnetwork.svg  (1369.7 KB)
  fig_k_subnetwork_4panel.png  (3254.9 KB)
  fig_k_subnetwork_4panel.svg  (5801.3 KB)


In [18]:
# Cell 7 — Batch: generate N=25, N=15, N=8 figure variants, organised by sub-folder.
#   Output layout (post-2026-05-15 reorg; BATCH_ROOT now defined in cell 1):
#     research/paper-materials/figures/results/k_subnetwork/N25/{fig_k{08,12,16,23},fig_k_subnetwork_4panel}.{png,svg}
#     research/paper-materials/figures/results/k_subnetwork/N15/...
#     research/paper-materials/figures/results/k_subnetwork/N8/...
import time

N_CHOICES = [25, 15, 8]
# BATCH_ROOT is defined in cell 1 as FIG_DIR / 'results' / 'k_subnetwork'
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Batch root: {BATCH_ROOT.relative_to(REPO)}")

def render_panel_for_n(N_choice, K, save_dir, global_top, abs_min, abs_max):
    k_idx = K_IDX[K]
    k_set = set(int(i) for i in k_idx)
    edges = [(i, j, v) for (i, j, v) in global_top if i in k_set and j in k_set]

    def _r(v):
        norm = (abs(v) - abs_min) / (abs_max - abs_min + 1e-12)
        return TUBE_RADIUS_MIN_MM + norm * (TUBE_RADIUS_MAX_MM - TUBE_RADIUS_MIN_MM)

    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex=CORTEX_STYLE, alpha=CORTEX_ALPHA, units='mm', size=(1100, 950),
    )
    for i in range(N_CH):
        in_K  = (i in k_set)
        color = ROI_COLOR[ROI_OF_IDX[i]] if in_K else (0.6, 0.6, 0.6)
        alpha = 1.0 if in_K else NONK_DOT_ALPHA
        r     = DOT_RADIUS_MM if in_K else DOT_NONK_RADIUS_MM
        b.plotter.add_mesh(pv.Sphere(center=mri_pos_dots[i], radius=r),
                            color=color, opacity=alpha, name=f"dot_{i}")
    label_pos = mri_pos_dots[list(k_idx)] + np.array([0, 0, LABEL_UP_MM])
    label_txt = [str(int(i) + 1) for i in k_idx]
    b.plotter.add_point_labels(
        label_pos, label_txt,
        font_size=22, text_color="black",
        point_color="white", point_size=1,
        show_points=False, always_visible=True, shape=None, margin=0,
        name=f"labels_{K}",
    )
    for (i, j, v) in edges:
        line = pv.Line(edge_pos[i], edge_pos[j], resolution=24).tube(radius=_r(v), n_sides=20)
        color = EDGE_COLOR_GAD_GT_HC if v > 0 else EDGE_COLOR_HC_GT_GAD
        b.plotter.add_mesh(line, color=color, opacity=EDGE_OPACITY, name=f"edge_{i}_{j}")
    try: b.show_view(view='rostral')
    except Exception: b.show_view()
    p = save_dir / f"k_subnet_K{K:02d}.png"
    b.save_image(str(p), mode='rgba')
    b.close()
    return p, edges

from matplotlib.lines import Line2D
t0 = time.time()
summary = {}
for N_choice in N_CHOICES:
    print(f"\n##### N = {N_choice} #####")
    out_dir = BATCH_ROOT / f"N{N_choice}"
    interm  = out_dir / "_intermediate"
    out_dir.mkdir(parents=True, exist_ok=True)
    interm.mkdir(parents=True, exist_ok=True)

    g_top = select_global_top_n(diff, N_choice)
    abs_vals_n = np.array([abs(v) for (_, _, v) in g_top])
    a_min, a_max = abs_vals_n.min(), abs_vals_n.max()
    print(f"  |M_diff| range: [{a_min:.5f}, {a_max:.5f}]")

    paths_n, edges_n = {}, {}
    for K in [23, 16, 12, 8]:
        p, e = render_panel_for_n(N_choice, K, interm, g_top, a_min, a_max)
        paths_n[K] = p
        edges_n[K] = e
        print(f"    K={K:2d}: {len(e):2d}/{N_choice} surviving edges")

    cropped_n = {K: crop_white(paths_n[K]) for K in [23, 16, 12, 8]}

    for K in [23, 16, 12, 8]:
        fig, ax = plt.subplots(figsize=(6, 5.5))
        ax.imshow(cropped_n[K])
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        ax.set_title(f"K = {K}   ({len(edges_n[K])}/{N_choice} top-decile edges)",
                     fontsize=14, fontweight='bold')
        p_out = out_dir / f"fig_k{K:02d}_subnetwork.png"
        fig.savefig(str(p_out), dpi=300, bbox_inches='tight', facecolor='white')
        fig.savefig(str(p_out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
        plt.close(fig)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
    for ax, K in zip(axes, [23, 16, 12, 8]):
        ax.imshow(cropped_n[K])
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        ax.set_title(f"K = {K}   ({len(edges_n[K])}/{N_choice} top-decile edges)",
                     fontsize=14, fontweight='bold')
    legend_handles = [
        Line2D([0],[0], color=EDGE_COLOR_GAD_GT_HC, lw=5, label='GAD > HC (anxiety-associated)'),
        Line2D([0],[0], color=EDGE_COLOR_HC_GT_GAD, lw=5, label='HC > GAD (healthy-dominant)'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=12, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f"Discriminative sub-network on cortex — global top-{N_choice} |M_diff| edges (line width ∝ |M_diff|)",
                 fontsize=14, y=1.0)
    plt.subplots_adjust(left=0.005, right=0.995, top=0.88, bottom=0.07, wspace=0.02)
    p_comp = out_dir / 'fig_k_subnetwork_4panel.png'
    fig.savefig(str(p_comp), dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(str(p_comp.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
    plt.close(fig)

    summary[N_choice] = {K: len(edges_n[K]) for K in [23, 16, 12, 8]}

print(f"\nTotal batch time: {time.time()-t0:.1f}s")
print(f"\nSurviving-edge counts (per N × K):")
print(f"  {'N':>4s}  {'K=23':>6s}  {'K=16':>6s}  {'K=12':>6s}  {'K=8':>6s}")
for N, counts in summary.items():
    print(f"  {N:>4d}  " + "  ".join(f"{counts[K]:>3d}/{N:<2d}" for K in [23,16,12,8]))

print(f"\nFiles per subfolder:")
for sub in sorted(BATCH_ROOT.iterdir()):
    if sub.is_dir() and sub.name.startswith('N'):
        pngs = sorted(sub.glob('fig_*.png'))
        svgs = sorted(sub.glob('fig_*.svg'))
        print(f"  {sub.relative_to(REPO)}/  ({len(pngs)} PNG + {len(svgs)} SVG)")

Batch root: research/paper-materials/figures/k_subnetwork

##### N = 25 #####
  |M_diff| range: [0.00371, 0.00669]


    K=23: 25/25 surviving edges


    K=16: 16/25 surviving edges


    K=12: 11/25 surviving edges


    K= 8:  5/25 surviving edges


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.



Total batch time: 50.6s

Surviving-edge counts (per N × K):
     N    K=23    K=16    K=12     K=8
    25   25/25   16/25   11/25    5/25
    15   15/15   12/15   10/15    5/15
     8    8/8     7/8     6/8     3/8 

Files per subfolder:
  research/paper-materials/figures/k_subnetwork/N15/  (5 PNG + 5 SVG)
  research/paper-materials/figures/k_subnetwork/N25/  (5 PNG + 5 SVG)
  research/paper-materials/figures/k_subnetwork/N8/  (5 PNG + 5 SVG)
